Auteur:     Eva Rombouts  
Datum:      23-12-2023  
Project:    GenCareAI  
Doel:       Verkennen en opschonen van de data die met de OpenAI API is gegenereerd.

In het 010 script zijn clientprofielen gemaakt voor 24 clienten met een vorm van dementie. De profielen zijn verhalend en ongestructureerd. Dit heb ik zo gelaten omdat het in de praktijk ook vaak zo gebeurt. 
In het 015 script zijn voor deze 24 clienten 10 weken aan rapportages gegenereerd. In deze rapportages is wel een vaste structuur aangebracht, maar het blijft nog een beetje spannend of die overal goed is gevolgd en consistent is. 
In dit script gaan we de opgeslagen json file omzetten naar twee datasets: df_clienten en df_rapportages.

Note to self:
Modellen:  
MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (multi NLI)  
wietsedv/bert-base-dutch-cased: Gebruikt door de michelin jongens. Op de site \<mask\>  
Robbert: pdelobelle/robbert-v2-dutch-base

In [ ]:
# Setup
import json
import re
import pandas as pd

# Parameters
seed = 6
filename_rapportages = '../zorgdata/appelboom_rapportages.json' # Bevat de opgeslagen JSON file
filename_clienten = '../zorgdata/appelboom_clienten.json' # Hier wordt de gecleande data naar geschreven

In [ ]:
# Lees de rapportages in
with open(filename_rapportages, 'r') as file:
    appelboom_rapportages = json.load(file)

De structuur van de appelboom_rapportages dictionary kan worden verbeterd en opgeschoond. 

Voor elke cliënt in appelboom_rapportages wordt een unieke ID gegenereerd in de vorm van kamer01, kamer02, enzovoorts. Deze unieke ID's vervangen de oorspronkelijke keys die beginnen met 'Naam: '.

De originele keys beginnen allemaal met 'Naam: '. Deze prefix wordt verwijderd om alleen de daadwerkelijke naam van de cliënt over te houden. Indien de naam 'Meneer' of 'Mevrouw' bevat, wordt dit gebruikt om het geslacht van de cliënt te bepalen. De naam wordt vervolgens opgeschoond door deze termen te verwijderen. Als de naam geen van deze aanduidingen bevat, wordt het geslacht als 'onbekend' gemarkeerd.

Voor elke cliënt worden de gegevens opnieuw gestructureerd. De 'Profiel' informatie wordt apart opgeslagen en alle rapportages die beginnen met 'Week' worden samengevoegd onder een aparte key 'rapportages'.

Het resultaat is een opgeschoonde en gestructureerde dictionary appelboom_clienten, waarbij elke cliënt een uniek ID heeft (het kamernummer) en de gegevens overzichtelijk zijn georganiseerd. 

In [ ]:
appelboom_clienten = {}

for i, (key, value) in enumerate(appelboom_rapportages.items()):
    uniek_id = f"kamer{i+1:02d}"  # Genereert ID's zoals 'kamer01', 'kamer02', etc.
    naam = key.replace('Naam: ', '') # Verwijder prefix
    naam_geslacht = naam.split(' ', 1)
    if 'Meneer' in naam_geslacht[0]:
        geslacht = 'man' 
        naam = naam_geslacht[1]
    elif 'Mevrouw' in naam_geslacht[0]:
        geslacht = 'vrouw' 
        naam = naam_geslacht[1]
    else:
        geslacht = 'onbekend'
    
    appelboom_clienten[uniek_id] = {
        "naam": naam,
        "geslacht": geslacht,
        "profiel": value.get("Profiel", ""),
        "rapportages": {k: v for k, v in value.items() if k.startswith("Week")}
    }


In [ ]:
# Maak een DataFrame voor de cliënten
df_clienten = pd.DataFrame.from_dict(appelboom_clienten, orient='index')
df_clienten.reset_index(inplace=True)
df_clienten.rename(columns={'index': 'kamernummer'}, inplace=True)
df_clienten.drop(columns=['rapportages'], inplace=True)


In [ ]:
print(df_clienten)

In [ ]:
# # Schrijf weg. 
# with open(filename_clienten, 'w') as file:
#     json.dump(appelboom_clienten, file) 

In [ ]:
# Voorbeeldje
profiel = appelboom_clienten['kamer01']['profiel']
print(profiel)

Je ziet dat de gegenereerde data lekker biased is. Meneer Pieterse was bankmanager. Als ik snel kijk zijn er ook twee architecten, allebei man. De dames zijn artistiek, doen vrijwilligerswerk, zijn familiemensen, op waren lerares...
Maar... het is wel realistisch.

In [ ]:
# voorbeeld rapportage_tekst
rapportage_tekst = appelboom_clienten['kamer01']['rapportages']['Week 1']
print(rapportage_tekst)

### Parsen van weekrapportages
Bij het genereren van de data is het llm gevraagd om rapportages van een week in gestructureerde vorm te maken. 
Deze worden gestart met ---StartRapportage--- en eindigen met --EindRapportage--- (let op. Per abuis start dit met 2 streepjes). 
vervolgens zijn dag/niveau/rapportage en onrustscore vaste kopjes. De onrustscore zit vaak zonder newline aan de rapportage geplakt. 

Allereerst willen we de weekrapportages splitsen in dagrapportages

In [ ]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


In [ ]:
aap = split_rapportage(rapportage_tekst)
# print (rapportage_tekst)

for ap in aap:
    print('-'*60)
    print (ap)

In [ ]:
def parse_dagrapportage(dagrapportage):
    # Zorg dat de onrustscore altijd op een nieuwe regel begint
    rapportage = re.sub(r'\n*(onrustscore)', '\nOnrustscore', dagrapportage, flags=re.IGNORECASE)
    rapportage_delen = re.split(r'\n', rapportage)

    parsed_data = {
        "dag": None,
        "niveau": None,
        "rapportage": None,
        "onrustscore": None
    }

    for deel in rapportage_delen:
        if deel.startswith('Dag:'):
            parsed_data["dag"] = deel[len('Dag:'):].strip().lower()
        elif deel.startswith('Niveau:'):
            parsed_data["niveau"] = deel[len('Niveau:'):].strip()
        elif deel.startswith('Rapportage:'):
            parsed_data["rapportage"] = deel[len('Rapportage:'):].strip()
        elif deel.startswith('Onrustscore:'):
            # Zet de onrustscore om naar een integer
            score = deel[len('Onrustscore:'):].strip()
            try:
                parsed_data["onrustscore"] = int(score)
            except ValueError:
                # Handel eventuele conversiefouten af
                parsed_data["onrustscore"] = None

    return parsed_data

In [ ]:
aap = 'Week3'
score = aap[len('week'):].strip()
print(score)

In [ ]:
noot = parse_dagrapportage(aap[0])
for key, value in noot.items():
    print (key + ": " + str(value))

In [ ]:
print(noot['rapportage'])

In [ ]:
print(aap[1])

In [ ]:
dagrapportages = []

for uniek_id, client_info in appelboom_clienten.items():
    for week, weekrapportage in client_info['rapportages'].items():
        weekno = week[len('week'):].strip()
        try:
            weekno = int(weekno)
        except ValueError:
            weekno = None
        dagrapportages_per_week = split_rapportage(weekrapportage)
        for dagrapportage in dagrapportages_per_week:
            rpt = parse_dagrapportage(dagrapportage)
            dagrapportages.append({
                "cliënt_id": uniek_id,
                "week": weekno,
                "dag": rpt['dag'],
                "niveau": rpt['niveau'],
                "rapportage": rpt['rapportage'],
                "onrustscore": rpt['onrustscore']
                # Vul hier aanvullende gegevens in, bijv. "datum", "niveau", "rapportage", "onrustscore"
            })

df_rapportages = pd.DataFrame(dagrapportages)


In [ ]:
print(df_rapportages)

In [ ]:
import locale
import datetime

# Stel de locale in op Nederlands
locale.setlocale(locale.LC_TIME, 'nl_NL')

datum = datetime.datetime.now()
weekdag = datum.strftime("%A").lower()  # Geeft de Nederlandse weekdag

print(weekdag)  # bijv. 'dinsdag' voor dinsdag


In [ ]:
df_rapportages['dag'].value_counts()

In [ ]:
df_rapportages['niveau'].value_counts()

In [ ]:
df_rapportages['onrustscore'].describe()

In [ ]:
df_rapportages.to_csv('../zorgdata/gci_rapportages.csv', index=False)

In [ ]:
df_clienten.to_csv('../zorgdata/gci_appelboom_profielen.csv', index=False)